# Lab 5 - Guided Lab: Facial Recognition

**Personal AWS account edition.** This notebook replaces the
Vocareum-provisioned `en_us/05-facedetection.ipynb`.

## Objectives
* Create a custom collection for Amazon Rekognition
* Add an image to a custom collection
* Detect known faces in an image
* View bounding boxes
* Delete the collection

## Cost

Essentially free. Amazon Rekognition's free tier covers 1,000 images per
month for 12 months; beyond that each API call costs about **$0.001**.
This whole notebook makes roughly 15 calls.

Face metadata storage is $0.00001 per face per month - and the last
cell deletes the collection anyway.

**The only thing that costs real money here is the notebook instance
you are running this on.** See the note below.

## You may not need SageMaker at all

Amazon Rekognition is a plain web API. This notebook only needs
`boto3`, `Pillow` and AWS credentials - **no machine learning runs on
your machine.** So it runs identically on:

* an AWS **CloudShell** session (free, already authenticated)
* your own laptop (`pip install boto3 pillow matplotlib`, `aws configure`)
* a SageMaker notebook instance (~$0.05/hr - the priciest option)

If you are watching costs, use CloudShell. The lab guide explains how.

## A note on responsible use

Facial recognition is not a neutral technology. Before running this,
it is worth stating plainly what the ethical issues are - your students
will encounter them in the real world:

* **Consent.** Faces are biometric data. In many jurisdictions
  (GDPR in the EU, BIPA in Illinois, and others) collecting or
  processing them without explicit informed consent is unlawful.
* **Accuracy is not uniform.** Published research, including the NIST
  Face Recognition Vendor Test, has repeatedly found that face
  recognition systems have **higher error rates for women and for
  people with darker skin**. A single global accuracy number hides
  this.
* **Consequences of error.** A false match in a policing or access
  control context can have severe consequences for a real person.

**This lab deliberately uses public-domain official portraits of public
figures**, not photos of students or private individuals. If you would
rather use your own photo, that is fine - it is your own face and your
own consent. **Do not upload photographs of other people without
asking them.**

A good discussion question for the end of the lab is included.

## Step 1 - Install and import dependencies

`boto3` is the AWS SDK for Python. It is pre-installed on SageMaker
notebook instances and in CloudShell.

In [ ]:
!pip install --quiet boto3 pillow matplotlib requests

import boto3, io, os, json, requests
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
%matplotlib inline

print('boto3', boto3.__version__)

## Step 2 - Connect to Amazon Rekognition

`boto3.client('rekognition')` picks up credentials automatically from
whatever environment you are in: the notebook instance's IAM role,
CloudShell's session, or your `~/.aws/credentials` file.

The cell below also verifies your credentials actually work before you
go any further - a much clearer failure than a confusing error five
cells later.

In [ ]:
REGION = 'us-east-1'

rekognition = boto3.client('rekognition', region_name=REGION)

# Confirm credentials are present and working
try:
    who = boto3.client('sts', region_name=REGION).get_caller_identity()
    print('Authenticated OK')
    print('  Account:', who['Account'])
    print('  Identity:', who['Arn'].split('/')[-1])
except Exception as e:
    print('CREDENTIALS PROBLEM:', type(e).__name__, e)
    print()
    print('Fix: on a SageMaker notebook the execution role needs')
    print('AmazonRekognitionFullAccess. On a laptop, run: aws configure')

## Step 3 - Get the images

We use **public-domain official portraits** published by the US federal
government. Works produced by US federal employees in the course of
their duties are in the public domain, so these are safe to use in
teaching material.

We download three images:

| File | Who | Role in the lab |
|---|---|---|
| `known_face.jpg` | Person A, portrait 1 | Indexed into the collection |
| `search_face.jpg` | Person A, portrait 2 (**different photo**) | Should **match** |
| `other_face.jpg` | Person B | Should **not** match |

Using two *different photographs of the same person* is the point of
the exercise. Matching an image against itself would prove nothing.

In [ ]:
UA = {'User-Agent': 'AWSAcademyLab/1.0 (educational use)'}
BASE = 'https://commons.wikimedia.org/wiki/Special:FilePath/'

IMAGES = {
    'known_face.jpg':  'Official_portrait_of_Barack_Obama.jpg',
    'search_face.jpg': 'President_Barack_Obama.jpg',
    'other_face.jpg':  'Joe_Biden_presidential_portrait.jpg',
}

for local, remote in IMAGES.items():
    if os.path.exists(local):
        print('Already have', local); continue
    url = BASE + remote + '?width=600'
    try:
        r = requests.get(url, headers=UA, timeout=60)
        r.raise_for_status()
        with open(local, 'wb') as f:
            f.write(r.content)
        print(f'Downloaded {local:18s} {len(r.content)/1024:6.0f} KB')
    except Exception as e:
        print(f'FAILED {local}: {type(e).__name__} {e}')

print()
print('If downloads failed, see the "use your own photos" cell below.')

### Alternative: use your own photos

If the downloads fail, or you would rather use your own face:

1. Take **two different photos of yourself** and one of a willing
   volunteer (or any second person who has agreed).
2. Upload them via the JupyterLab file browser's **upload** (↑) button.
3. Rename them to `known_face.jpg`, `search_face.jpg`, `other_face.jpg`
   - or edit the filenames in the cells below.

JPEG or PNG, under 5 MB, and the face should be reasonably clear and
front-facing.

In [ ]:
# Display the three images side by side
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
titles = ['known_face.jpg\n(indexed)',
          'search_face.jpg\n(should MATCH)',
          'other_face.jpg\n(should NOT match)']

for ax, (fn, title) in zip(axes, zip(IMAGES.keys(), titles)):
    if os.path.exists(fn):
        ax.imshow(Image.open(fn))
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Step 4 - Detect faces and draw bounding boxes

Before doing any *recognition*, let us do *detection*: find where the
faces are, and what Rekognition can tell us about them.

`detect_faces` with `Attributes=['ALL']` returns a bounding box plus
estimated attributes for every face it finds.

**Understanding the bounding box** - this trips people up. Rekognition
returns box coordinates as **ratios of the image size (0 to 1)**, not
pixels. That way the numbers stay valid whatever resolution you use.
To draw the box you multiply by the actual width and height:

```
left   = box['Left']   * image_width
top    = box['Top']    * image_height
width  = box['Width']  * image_width
height = box['Height'] * image_height
```

In [ ]:
def detect_and_draw(filename):
    with open(filename, 'rb') as f:
        image_bytes = f.read()

    response = rekognition.detect_faces(
        Image={'Bytes': image_bytes},
        Attributes=['ALL'],
    )

    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    W, H = img.size
    draw = ImageDraw.Draw(img)

    for face in response['FaceDetails']:
        box = face['BoundingBox']
        left, top = box['Left'] * W, box['Top'] * H
        width, height = box['Width'] * W, box['Height'] * H
        draw.rectangle([left, top, left + width, top + height],
                       outline='lime', width=4)

    return img, response['FaceDetails']


img, details = detect_and_draw('known_face.jpg')

plt.figure(figsize=(6, 7))
plt.imshow(img); plt.axis('off')
plt.title('detect_faces - bounding box'); plt.show()

print(f'Faces found: {len(details)}')

### What Rekognition thinks it sees

`Attributes=['ALL']` returns estimated age range, emotions, whether
the eyes are open, presence of glasses or a beard, and more - each with
a **confidence** score.

**Read these critically.** Confidence is the model's self-reported
certainty, not a measure of truth. Emotion estimates in particular are
inferences from facial muscle positions, and the scientific basis for
mapping expressions onto felt emotions is genuinely contested. Treat
them as "this face resembles patterns labelled *happy* in training
data" - not "this person is happy."

In [ ]:
face = details[0]

print(f"Overall confidence this is a face: {face['Confidence']:.2f}%")
print(f"Estimated age range: {face['AgeRange']['Low']}-{face['AgeRange']['High']}")
print(f"Smiling:    {face['Smile']['Value']}   ({face['Smile']['Confidence']:.1f}%)")
print(f"Eyeglasses: {face['Eyeglasses']['Value']}   ({face['Eyeglasses']['Confidence']:.1f}%)")
print(f"Beard:      {face['Beard']['Value']}   ({face['Beard']['Confidence']:.1f}%)")
print(f"Eyes open:  {face['EyesOpen']['Value']}   ({face['EyesOpen']['Confidence']:.1f}%)")
print()
print('Top emotions (as estimated by the model):')
for e in sorted(face['Emotions'], key=lambda x: -x['Confidence'])[:3]:
    print(f"  {e['Type']:12s} {e['Confidence']:5.1f}%")
print()
print('Head pose (degrees):')
p = face['Pose']
print(f"  Pitch {p['Pitch']:6.1f}  Roll {p['Roll']:6.1f}  Yaw {p['Yaw']:6.1f}")

## Step 5 - Create a collection

A **collection** is a server-side container of face vectors. When you
add a face to a collection, Rekognition does **not** store the image -
it stores a mathematical representation (a feature vector) that can be
compared against other faces.

That distinction matters:

* You cannot retrieve the original photo from a collection.
* You can still match a person, so the data is still biometric data and
  still regulated.

In [ ]:
COLLECTION_ID = 'lab5-faces'

# Remove any collection left over from a previous run
try:
    rekognition.delete_collection(CollectionId=COLLECTION_ID)
    print('Deleted a pre-existing collection with the same name')
except rekognition.exceptions.ResourceNotFoundException:
    pass

response = rekognition.create_collection(CollectionId=COLLECTION_ID)
print('Created collection:', COLLECTION_ID)
print('  Status code:', response['StatusCode'])
print('  ARN:', response['CollectionArn'])

In [ ]:
# List every collection in this account and region
print('Collections in', REGION, ':')
for c in rekognition.list_collections()['CollectionIds']:
    print('  -', c)

## Step 6 - Add a face to the collection

`index_faces` detects faces in the image and stores their vectors.

**`ExternalImageId` is the label you will get back on a match.** This
is how you attach an identity to a face vector - a name, an employee
number, a database key. Choose it deliberately; it is the only thing
that comes back when you later search.

> It accepts letters, numbers, and `_ . : - ` only. No spaces.

In [ ]:
with open('known_face.jpg', 'rb') as f:
    known_bytes = f.read()

response = rekognition.index_faces(
    CollectionId=COLLECTION_ID,
    Image={'Bytes': known_bytes},
    ExternalImageId='Person_A',      # the label returned on a match
    DetectionAttributes=['DEFAULT'],
    MaxFaces=1,
    QualityFilter='AUTO',
)

for record in response['FaceRecords']:
    face = record['Face']
    print('Indexed a face:')
    print('  FaceId          :', face['FaceId'])
    print('  ExternalImageId :', face['ExternalImageId'])
    print(f"  Confidence      : {face['Confidence']:.2f}%")
    print('  BoundingBox     :',
          {k: round(v, 3) for k, v in face['BoundingBox'].items()})

if response.get('UnindexedFaces'):
    print()
    print('Faces rejected:', len(response['UnindexedFaces']))
    for u in response['UnindexedFaces']:
        print('  Reasons:', u['Reasons'])

## Step 7 - List the faces in the collection

In [ ]:
faces = rekognition.list_faces(CollectionId=COLLECTION_ID)['Faces']

print(f'{len(faces)} face(s) in collection "{COLLECTION_ID}":')
for f in faces:
    print(f"  {f['ExternalImageId']:12s}  FaceId={f['FaceId']}")

## Step 8 - Search for a known face ← the point of the lab

`search_faces_by_image` takes a **new** photo, extracts its face
vector, and compares it against every vector in the collection.

**`FaceMatchThreshold`** is the minimum similarity (0-100) to count as
a match. The default is 80. This is the dial that trades the two error
types against each other:

| Threshold | Effect |
|---|---|
| **Low** (e.g. 70) | More matches found; more **false positives** - wrong people matched |
| **High** (e.g. 95) | Fewer matches; more **false negatives** - right person missed |

There is no universally correct value. In an access-control system a
false positive lets the wrong person in; in a photo-tagging feature it
is a minor annoyance. **The right threshold depends on the cost of each
mistake** - the same lesson as the classification threshold in Lab 3.6.

In [ ]:
with open('search_face.jpg', 'rb') as f:
    search_bytes = f.read()

response = rekognition.search_faces_by_image(
    CollectionId=COLLECTION_ID,
    Image={'Bytes': search_bytes},
    FaceMatchThreshold=80,
    MaxFaces=5,
)

matches = response['FaceMatches']
print(f'Searched with a DIFFERENT photo of the same person.')
print(f'Matches found: {len(matches)}')
print()

if matches:
    for m in matches:
        print(f"  MATCH: {m['Face']['ExternalImageId']}")
        print(f"    Similarity : {m['Similarity']:.2f}%")
        print(f"    Confidence : {m['Face']['Confidence']:.2f}%")
else:
    print('  No match above the threshold.')

### Draw the bounding box of the searched face

`SearchedFaceBoundingBox` tells you *which* face in the submitted image
was used for the search - useful when the photo contains several people.

In [ ]:
img = Image.open(io.BytesIO(search_bytes)).convert('RGB')
W, H = img.size
draw = ImageDraw.Draw(img)

box = response['SearchedFaceBoundingBox']
left, top = box['Left'] * W, box['Top'] * H
width, height = box['Width'] * W, box['Height'] * H

colour = 'lime' if matches else 'red'
draw.rectangle([left, top, left + width, top + height],
               outline=colour, width=5)

label = (f"{matches[0]['Face']['ExternalImageId']} "
         f"({matches[0]['Similarity']:.1f}%)") if matches else 'NO MATCH'
draw.text((left, max(0, top - 22)), label, fill=colour)

plt.figure(figsize=(6, 7))
plt.imshow(img); plt.axis('off')
plt.title(f'search_faces_by_image → {label}'); plt.show()

## Step 9 - The negative test

A recognition system that matches *everyone* is useless. Test that a
**different person does not match.**

This is the step the original lab omits, and it is the one that proves
the system actually works. Always test the negative case.

In [ ]:
with open('other_face.jpg', 'rb') as f:
    other_bytes = f.read()

response_other = rekognition.search_faces_by_image(
    CollectionId=COLLECTION_ID,
    Image={'Bytes': other_bytes},
    FaceMatchThreshold=80,
    MaxFaces=5,
)

other_matches = response_other['FaceMatches']
print('Searched with a photo of a DIFFERENT person.')
print(f'Matches found: {len(other_matches)}')
print()
if other_matches:
    for m in other_matches:
        print(f"  Unexpected match: {m['Face']['ExternalImageId']} "
              f"at {m['Similarity']:.2f}% - this is a FALSE POSITIVE")
else:
    print('  Correctly found no match. The system discriminates properly.')

### How similarity varies with the threshold

Re-run the same two searches at several thresholds. This makes the
trade-off concrete rather than abstract.

In [ ]:
import pandas as pd

rows = []
for threshold in [50, 60, 70, 80, 90, 95, 99]:
    same = rekognition.search_faces_by_image(
        CollectionId=COLLECTION_ID, Image={'Bytes': search_bytes},
        FaceMatchThreshold=threshold, MaxFaces=5)['FaceMatches']
    diff = rekognition.search_faces_by_image(
        CollectionId=COLLECTION_ID, Image={'Bytes': other_bytes},
        FaceMatchThreshold=threshold, MaxFaces=5)['FaceMatches']
    rows.append({
        'threshold': threshold,
        'same person matched': 'YES' if same else 'no',
        'similarity': round(same[0]['Similarity'], 1) if same else None,
        'other person matched': 'FALSE POSITIVE' if diff else 'no',
    })

pd.DataFrame(rows).set_index('threshold')

**Read the table.** Find the range of thresholds where the same person
still matches but the different person does not - that is your usable
operating window. Notice how narrow or wide it is, and remember it was
measured on exactly **two** people.

A real evaluation needs thousands of faces, and must report accuracy
**broken down by demographic group** - not as a single number.

## Step 10 - Clean up: delete the collection

Face vectors persist until deleted, and they are biometric data. There
is no reason to leave them lying in an account after a lab.

The storage cost is trivially small ($0.00001/face/month); **deleting
is about data hygiene, not cost.**

In [ ]:
rekognition.delete_collection(CollectionId=COLLECTION_ID)
print('Deleted collection:', COLLECTION_ID)

remaining = rekognition.list_collections()['CollectionIds']
print()
print('Collections remaining in this account/region:',
      remaining if remaining else 'none')

In [ ]:
# Optional: also remove the downloaded images from the instance
for fn in IMAGES:
    if os.path.exists(fn):
        os.remove(fn)
        print('Removed', fn)

## Conclusion

You have:
* Created a custom Amazon Rekognition collection
* Added a face to it with an `ExternalImageId` label
* Listed the faces in the collection
* Matched a **different photograph** of the same person
* Verified a different person does **not** match
* Seen how `FaceMatchThreshold` trades false positives against false
  negatives
* Drawn bounding boxes from ratio coordinates
* Deleted the collection

### Discussion questions

1. Your two-person test worked. Why is that *not* evidence the system
   is accurate? *(Sample size of two; no demographic breakdown; ideal
   lighting; posed studio portraits.)*
2. You are building door access for an office. Do you set the threshold
   at 70 or 99, and what goes wrong with each?
3. NIST testing has found error rates vary substantially across
   demographic groups. What would you have to measure before deploying
   this, and what would you do if accuracy differed by group?
4. The collection stores vectors, not images. Does that make it
   exempt from data-protection law? *(No - it is still biometric data
   that identifies a person.)*

### Cost checkpoint

* Collection deleted ✓
* Rekognition calls made: ~15, well inside the 1,000/month free tier
* **If you are on a SageMaker notebook instance, STOP IT NOW** -
  it is the only thing here that bills by the hour.